# I. OptiTrack

This section provides a concise guide on how to use **OptiTrack** for motion capture.
For more detailed information, refer to the official documentation:  
https://docs.optitrack.com/v3.0/motive

---

### 1. Camera Installation

- Ensure that all angles are well covered.
- Center the cameras on the robot.
- Avoid closing the diaphragm too much. Even if it introduces more noise, the markers will be significantly more visible.
- Ensure proper focus, especially for distant cameras.
- If a camera detects nothing:
  - First check the aperture.
  - If the issue persists, change the camera position.
- For calibration, strictly follow the instructions provided on the OptiTrack website.

---

### 2. Recording

- No specific constraints or special steps are required during recording.

---

### 3. Editing and Exporting

- Switch to **Edit Mode** by selecting the *Edit* layout.
- Select only the frames of interest (i.e., the frames during which the wing is moving).
- Create a **Rigid Body**:
  - Find a frame where all markers are visible.
  - Select the markers.
  - In the Builder panel, click **Create**.
- Label each marker using the format `Column–Row` (e.g., `A1`, `B3`, etc.), starting from `A1`.
  - This step must be done **manually**.
- Use **Auto-Label** to associate markers with each other (available in the *Label* window).
- Verify each marker trajectory for completeness and smoothness:
  - Click on a marker and inspect its trajectory in the **Graph View**.

#### Trajectory Corrections

- **Incomplete trajectories**:
  - Fill gaps using the *Cubic* method (recommended and generally more reliable).
  - Use the *Comparative* method only for large gaps, and select only the 2–3 closest markers.
- **Spikes or abnormal data**:
  - Cut the corrupted portion of the curve.
  - Reconstruct it using the *Cubic* method.
- **Smoothing**:
  - Apply smoothing to the trajectory.
  - Determine the appropriate smoothing frequency by testing multiple values.
  - The goal is a smooth trajectory without excessively deforming the wing shape or motion.

---

### 4. Exporting Data

- Export the labeled marker data to a CSV file:
  - Click **File → Export Tracking Data**.
  - Set the *Start Frame* and *Last Frame* to include only the frames of interest.
  - Export the data.


# II. Marker Placement

Correct marker placement is critical to ensure accurate reconstruction of the wing geometry and motion.

- Markers must be aligned **column-wise** along the span.
  - Perfect alignment in rows is **not required** and is not an issue.
- Place **at least two markers per column**.
- For each column:
  - Place **one marker on the leading edge**.
  - Place **one marker at the wingtip**.
  - Add the remaining markers afterward, distributed along the chord.
- Ensure all markers remain visible throughout the motion to avoid trajectory gaps.

### Example Setup

![Marker placement setup](setup.png)




# III. PteraSoftware

This section explains how to import **OptiTrack** CSV files into **PteraSoftware** and use them for aircraft simulations. An example script is available here:
`examples\Optitrack_unsteady_ring_vortex_lattice_method_solver_variable.py`

---

### 1. Create Variables

* Define the path to the OptiTrack file:

```python
optitrack_file = "path/to/optitrack_file.csv"
```

* Define the list of trackers :

```python
list_trackers = [
    "A1", "A2", "A3", "B1", "B2", "B3", "B4", "B5",
    "C1", "C2", "C3", "C4", "C5", "D1", "D2", "D3", "D4", "D5",
    "E1", "E2", "E3", "E4", "E5", "F1", "F2", "F3", "F4", "F5", 
    "G1", "G2", "G3", "G4", "G5", "H1", "H2", "H3", "H4",
    "I1", "I2"
]
```

* Extract columns and define data:

```python
columns = ps.geometry.airfoil_creation.extract_columns(list_trackers)
Nb_columns = len(columns)
data = ps.geometry.airfoil_creation.load_data(optitrack_file, list_trackers, right=False) * 10**-3  # Convert from mm to m
```

---

### 2. Create Airfoils at t0

* Create an empty dictionary for airfoils:

```python
airfoils_0 = {}
```

* Loop through columns to populate airfoils:

```python
for column in columns:
    airfoils_0[column] = ps.geometry.airfoil_creation.Real_Airfoil(
        data=data,
        step=0,
        column=column,
        list_trackers=list_trackers,
    )
```

---

### 3. Create the Aircraft

* Assign each airfoil to its wing cross-section. Handle the last column exception:

```python
example_airplane = ps.geometry.airplane.Airplane(
    wings=[
        ps.geometry.wing.Wing(
            wing_cross_sections=[
                ps.geometry.wing_cross_section.WingCrossSection(
                    num_spanwise_panels = None if column == columns[-1] else 1, # Last wing cross-section has no panels 
                    chord=airfoils_0[column].get_chord_length(), 
                    Lp_Wcsp_Lpp=(0,0,0) if column == 'A' else airfoils_0[column].get_relative_transform()[0],
                    angles_Wcsp_to_Wcs_ixyz=(0,0,0) if column == 'A' else airfoils_0[column].get_relative_transform()[1],
                    control_surface_symmetry_type="symmetric",
                    control_surface_hinge_point=0.75,
                    control_surface_deflection=0.0,
                    spanwise_spacing=None if column == columns[-1] else "uniform", # Last wing cross-section has no panels
                    airfoil=ps.geometry.airfoil.Airfoil(
                        name=f"column_{column}_airfoil",  
                        outline_A_lp=airfoils_0[column].get_airfoil_shape(),
                        resample=True,
                        n_points_per_side=400,
                        data=data,    # Pass the data to the Airfoil
                        column=column,        # Pass the column to the Airfoil
                        list_trackers=list_trackers,       # Pass the list of trackers to the Airfoil
                    ),
                )
                for column in columns
            ],
            name="Main Wing",
            Ler_Gs_Cgs= np.array([0.0, 0.025, 0.0]),
            angles_Gs_to_Wn_ixyz= np.array([4, 0.0, 0.0]),
            symmetric=True,
            mirror_only=False,
            symmetryNormal_G=(0.0, 0.0001, 0.0),
            symmetryPoint_G_Cg=(0.0, 0.0, 0.0),
            num_chordwise_panels=6,
            chordwise_spacing="uniform",
        )
    ],
    name="Example Airplane",
    Cg_E_CgP1=(0.0, 0.0, 0.0),
    angles_E_to_B_izyx=(0.0, 0.0, 0.0),
    weight=5,
    s_ref=None,
    c_ref=None,
    b_ref=None,
)
```

---

### 4. Create Wing Cross-Section Movements

* Initialize empty movement lists:

```python
main_wing_cross_section_movement = [None] * Nb_columns
reflected_main_wing_cross_section_movement = [None] * Nb_columns
```

* Loop through columns to create movements using OptiTrack data:

```python
for i in range(Nb_columns):
    main_wing_cross_section_movement[i] = (
        ps.movements.wing_cross_section_movement.WingCrossSectionMovement(
            base_wing_cross_section=example_airplane.wings[0].wing_cross_sections[i],
            ampLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
            periodLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
            spacingLp_Wcsp_Lpp=("sine", "sine", "sine"),
            phaseLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
            ampAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
            periodAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
            spacingAngles_Wcsp_to_Wcs_ixyz=("sine", "sine", "sine"),
            phaseAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
            optitrack=True
        )
    )

for i in range(Nb_columns):
    reflected_main_wing_cross_section_movement[i] = (
        ps.movements.wing_cross_section_movement.WingCrossSectionMovement(
            base_wing_cross_section=example_airplane.wings[1].wing_cross_sections[i],
            ampLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
            periodLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
            spacingLp_Wcsp_Lpp=("sine", "sine", "sine"),
            phaseLp_Wcsp_Lpp=(0.0, 0.0, 0.0),
            ampAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
            periodAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
            spacingAngles_Wcsp_to_Wcs_ixyz=("sine", "sine", "sine"),
            phaseAngles_Wcsp_to_Wcs_ixyz=(0.0, 0.0, 0.0),
            optitrack=True
        )
    )
```

---

### 5. Airplane Movement and Solver

* Define **WingMovements** : 
```python
main_wing_movement = ps.movements.wing_movement.WingMovement(
    base_wing=example_airplane.wings[0],
    wing_cross_section_movements= main_wing_cross_section_movement,
    ampLer_Gs_Cgs=(0.0, 0.0, 0.0),
    periodLer_Gs_Cgs=(0.0, 0.0, 0.0),
    spacingLer_Gs_Cgs=("sine", "sine", "sine"),
    phaseLer_Gs_Cgs=(0.0, 0.0, 0.0),
    ampAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0), 
    periodAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0), 
    spacingAngles_Gs_to_Wn_ixyz=("sine", "sine", "sine"),
    phaseAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
    optitrack = True  # Put to True to use OptiTrack data. The other parameters will be ignored except for base_wing
)
reflected_main_wing_movement = ps.movements.wing_movement.WingMovement(
    base_wing=example_airplane.wings[1],
    wing_cross_section_movements=reflected_main_wing_cross_section_movement,
    ampLer_Gs_Cgs=(0.0, 0.0, 0.0),
    periodLer_Gs_Cgs=(0.0, 0.0, 0.0),
    spacingLer_Gs_Cgs=("sine", "sine", "sine"),
    phaseLer_Gs_Cgs=(0.0, 0.0, 0.0),
    ampAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),  
    periodAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0), 
    spacingAngles_Gs_to_Wn_ixyz=("sine", "sine", "sine"),
    phaseAngles_Gs_to_Wn_ixyz=(0.0, 0.0, 0.0),
    optitrack = True  # Put to True to use OptiTrack data. The other parameters will be ignored except for base_wing
)
```


* Define **AirplaneMovement**, **OperatingPoint**, and combine into a **Movement** object.
* Define **UnsteadyProblem** and the solver:

```python
example_problem = ps.problems.UnsteadyProblem(movement=movement)
example_solver = ps.unsteady_ring_vortex_lattice_method.UnsteadyRingVortexLatticeMethodSolver(
    unsteady_problem=example_problem
)
example_solver.run(logging_level="Warning", prescribed_wake=True)
```


### 6. Plot Results

* Animate the airplane:

```python
ps.output.animate(
    unsteady_solver=example_solver,
    scalar_type="lift",
    show_wake_vortices=True,
    save=True
)
```

![Animation](Animate.webp)

* Plot forces and moments vs time:

```python
ps.output.plot_results_versus_time(unsteady_solver=example_solver, show=True)
```

![Forces Airplane](force_airplane.png)

* Plot wing loads:

```python
ps.output.plot_wing_loads_versus_time(unsteady_solver=example_solver, show=True)
```

![Momments Wing](moment_wing.png)

* Print total results:

```python
ps.output.print_results(example_solver)
```

---

### 7. Compare to a Simulated Wing

* Create a fully simulated airplane with the same characteristics:

```python
analysis = ps.differential_measures.Analysis(example_solver, 4)  # 4 Hz flapping
```

* Functions available:

  * Dynamic wing comparison: `analysis.dynamic_wing()`

![Dynamic](Dynamic.gif)

  * 3D trajectory of a point: `analysis.plot_trajectory_3d(0.8, 0.5)`

![3d](3d_position.png)

  * Average difference in position vs time: `analysis.plot_difference_position_versus_time()`

![Position_vs_time](position_vs_time.png)

  * Wing section: `analysis.plot_section(5)`

![Section](Wing_section.png)

  * Panel forces: `analysis.plot_panel_forces(0.8, 0.9)`

![Force\_panel](Force_panel.png)

  * Forces and moments vs time: `analysis.plot_forces()`

![Force\_airplane\_dif](Force_airplane_dif.png)

  * Differential visualization:

```python
ps.output.animate(
    unsteady_solver=example_solver,
    scalar_type="difference position",
    show_wake_vortices=True,
    fake_solver=analysis.fake_solver,
    save=True
)
```

![Differential](Difference.webp)
